In [7]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [44]:
import joblib
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    confusion_matrix, average_precision_score
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import RandomizedSearchCV

In [9]:
X_train = pd.read_csv("X_train.csv")
X_test = pd.read_csv("X_test.csv")
y_train = pd.read_csv("y_train.csv").squeeze()
y_test = pd.read_csv("y_test.csv").squeeze()

In [10]:
results = []

In [11]:
def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    pr_auc = average_precision_score(y_test, y_proba)
    cm = confusion_matrix(y_test, y_pred)

    print(f"--- {name} ---")
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f} | PR-AUC: {pr_auc:.4f}")
    print("Confusion Matrix:\n", cm)
    print()
    return {"model": name, "precision": precision, "recall": recall, "f1": f1, "pr_auc": pr_auc}

In [12]:
rf_baseline = joblib.load("baseline_rf.joblib")
results.append(evaluate_model("Random Forest (baseline)", rf_baseline, X_test, y_test))

--- Random Forest (baseline) ---
Precision: 0.9569 | Recall: 0.7817 | F1: 0.8605 | PR-AUC: 0.8031
Confusion Matrix:
 [[84971     5]
 [   31   111]]



In [13]:
xgb_plain = XGBClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    n_jobs=-1,
    eval_metric="aucpr"
)
xgb_plain.fit(X_train, y_train)
results.append(evaluate_model("XGBoost (plain)", xgb_plain, X_test, y_test))

--- XGBoost (plain) ---
Precision: 0.9619 | Recall: 0.7113 | F1: 0.8178 | PR-AUC: 0.8249
Confusion Matrix:
 [[84972     4]
 [   41   101]]



In [14]:
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
spw = neg / pos
print("scale_pos_weight:", round(spw, 1))

scale_pos_weight: 599.0


In [15]:
xgb_weighted = XGBClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    scale_pos_weight=spw,
    random_state=42,
    n_jobs=-1,
    eval_metric="aucpr"
)
xgb_weighted.fit(X_train, y_train)
results.append(evaluate_model("XGBoost (scale_pos_weight)", xgb_weighted, X_test, y_test))

--- XGBoost (scale_pos_weight) ---
Precision: 0.9310 | Recall: 0.7606 | F1: 0.8372 | PR-AUC: 0.8137
Confusion Matrix:
 [[84968     8]
 [   34   108]]



In [16]:
lgbm_plain = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
lgbm_plain.fit(X_train, y_train)
results.append(evaluate_model("LightGBM (plain)", lgbm_plain, X_test, y_test))

--- LightGBM (plain) ---
Precision: 0.0814 | Recall: 0.0493 | F1: 0.0614 | PR-AUC: 0.0056
Confusion Matrix:
 [[84897    79]
 [  135     7]]



In [17]:
lgbm_weighted = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.1,
    scale_pos_weight=spw,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
lgbm_weighted.fit(X_train, y_train)
results.append(evaluate_model("LightGBM (scale_pos_weight)", lgbm_weighted, X_test, y_test))

--- LightGBM (scale_pos_weight) ---
Precision: 0.0279 | Recall: 0.8310 | F1: 0.0540 | PR-AUC: 0.0235
Confusion Matrix:
 [[80865  4111]
 [   24   118]]



In [18]:
comparison_df = pd.DataFrame(results).sort_values("pr_auc", ascending=False).reset_index(drop=True)

In [19]:
comparison_df

,model,precision,recall,f1,pr_auc
0,XGBoost (plain),0.961905,0.711268,0.817814,0.824890
1,XGBoost (scale_pos_weight),0.931034,0.760563,0.837209,0.813679
2,Random Forest (baseline),0.956897,0.781690,0.860465,0.803127
3,LightGBM (scale_pos_weight),0.027903,0.830986,0.053992,0.023474
4,LightGBM (plain),0.081395,0.049296,0.061404,0.005598


In [20]:
proba_tr = lgbm_plain.predict_proba(X_train)[:, 1]
proba_te = lgbm_plain.predict_proba(X_test)[:, 1]

print("Train PR-AUC:", average_precision_score(y_train, proba_tr))
print("Test PR-AUC :", average_precision_score(y_test, proba_te))
print(pd.Series(proba_te).describe())

Train PR-AUC: 0.011959891697325354
Test PR-AUC : 0.00559848028009162
count    85118.00000
mean         0.00101
std          0.03177
min          0.00000
25%          0.00000
50%          0.00000
75%          0.00000
max          1.00000
dtype: float64


In [21]:
import lightgbm as lgb
print("LightGBM version:", lgb.__version__)

LightGBM version: 4.6.0


In [22]:
lgbm_dbg = LGBMClassifier(n_estimators=200, learning_rate=0.1, random_state=42, n_jobs=-1)
lgbm_dbg.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train)],
    eval_metric="average_precision",
    callbacks=[lgb.log_evaluation(period=25)]
)

[25]	training's average_precision: 0.773197	training's binary_logloss: 0.0161416
[50]	training's average_precision: 0.693281	training's binary_logloss: 0.0458632
[75]	training's average_precision: 0.732306	training's binary_logloss: 0.0254653
[100]	training's average_precision: 0.631721	training's binary_logloss: 0.0361787
[125]	training's average_precision: 0.516458	training's binary_logloss: 0.099184
[150]	training's average_precision: 0.105193	training's binary_logloss: 0.141713
[175]	training's average_precision: 0.064174	training's binary_logloss: 0.087917
[200]	training's average_precision: 0.064174	training's binary_logloss: 0.087917


LGBMClassifier(n_estimators=200, n_jobs=-1, random_state=42)

In [23]:
lgbm_np = LGBMClassifier(n_estimators=200, learning_rate=0.1, random_state=42, n_jobs=-1, verbose=-1)
lgbm_np.fit(X_train.values, y_train.values)
proba = lgbm_np.predict_proba(X_train.values)[:, 1]
print("Train PR-AUC (numpy input):", average_precision_score(y_train, proba))

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Train PR-AUC (numpy input): 0.011959891697325354


In [24]:
raw = lgbm_np.predict(X_train.values, raw_score=True)
print("raw score min/max:", raw.min(), raw.max())
print("PR-AUC on raw scores:", average_precision_score(y_train, raw))
print("Share of rows with proba exactly 0 or 1:", ((proba == 0) | (proba == 1)).mean())

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


raw score min/max: -489478.355649107 68964.54386916463
PR-AUC on raw scores: 0.06417395049321835
Share of rows with proba exactly 0 or 1: 0.9998640538145492


In [25]:
lgbm_stable = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.03,
    num_leaves=15,
    reg_lambda=5.0,
    max_delta_step=2.0,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
lgbm_stable.fit(X_train, y_train)
results.append(evaluate_model("LightGBM (stabilised)", lgbm_stable, X_test, y_test))

--- LightGBM (stabilised) ---
Precision: 0.9381 | Recall: 0.7465 | F1: 0.8314 | PR-AUC: 0.8161
Confusion Matrix:
 [[84969     7]
 [   36   106]]



In [26]:
comparison_df = pd.DataFrame(results).sort_values("pr_auc", ascending=False).reset_index(drop=True)

In [27]:
comparison_df

,model,precision,recall,f1,pr_auc
0,XGBoost (plain),0.961905,0.711268,0.817814,0.824890
1,LightGBM (stabilised),0.938053,0.746479,0.831373,0.816054
2,XGBoost (scale_pos_weight),0.931034,0.760563,0.837209,0.813679
3,Random Forest (baseline),0.956897,0.781690,0.860465,0.803127
4,LightGBM (scale_pos_weight),0.027903,0.830986,0.053992,0.023474
5,LightGBM (plain),0.081395,0.049296,0.061404,0.005598


In [28]:
joblib.dump(xgb_plain, "xgb_plain.joblib")
joblib.dump(xgb_weighted, "xgb_weighted.joblib")
joblib.dump(lgbm_stable, "lgbm_stabilised.joblib")

['lgbm_stabilised.joblib']

In [29]:
iso = IsolationForest(
    n_estimators=200,
    contamination=float(y_train.mean()),
    random_state=42,
    n_jobs=-1
)
iso.fit(X_train)

IsolationForest(contamination=0.0016665995327479256, n_estimators=200,
                n_jobs=-1, random_state=42)

In [32]:
anomaly_score = -iso.score_samples(X_test)
y_pred_iso = (iso.predict(X_test) == -1).astype(int)

precision = precision_score(y_test, y_pred_iso)
recall = recall_score(y_test, y_pred_iso)
f1 = f1_score(y_test, y_pred_iso)
pr_auc = average_precision_score(y_test, anomaly_score)

print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f} | PR-AUC: {pr_auc:.4f}")
print(confusion_matrix(y_test, y_pred_iso))

results.append({
    "model": "Isolation Forest (unsupervised)",
    "precision": precision, "recall": recall, "f1": f1, "pr_auc": pr_auc
})

Precision: 0.2153 | Recall: 0.2183 | F1: 0.2168 | PR-AUC: 0.1083
[[84863   113]
 [  111    31]]


In [33]:
comparison_df = pd.DataFrame(results).sort_values("pr_auc", ascending=False).reset_index(drop=True)

In [34]:
comparison_df

,model,precision,recall,f1,pr_auc
0,XGBoost (plain),0.961905,0.711268,0.817814,0.824890
1,LightGBM (stabilised),0.938053,0.746479,0.831373,0.816054
2,XGBoost (scale_pos_weight),0.931034,0.760563,0.837209,0.813679
3,Random Forest (baseline),0.956897,0.781690,0.860465,0.803127
4,Isolation Forest (unsupervised),0.215278,0.218310,0.216783,0.108285
5,LightGBM (scale_pos_weight),0.027903,0.830986,0.053992,0.023474
6,LightGBM (plain),0.081395,0.049296,0.061404,0.005598


In [37]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [36]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [38]:
models_to_test = {
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "XGBoost (plain)": XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=42, n_jobs=-1),
    "XGBoost (weighted)": XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=6, scale_pos_weight=spw, random_state=42, n_jobs=-1),
    "LightGBM (stable)": LGBMClassifier(n_estimators=300, learning_rate=0.03, num_leaves=15, reg_lambda=5.0, max_delta_step=2.0, random_state=42, n_jobs=-1, verbose=-1)
}

In [40]:
cv_results = []

In [41]:
for name, model in models_to_test.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="average_precision", n_jobs=-1)
    cv_results.append({
        "model": name,
        "mean_pr_auc": scores.mean(),
        "std_pr_auc": scores.std(),
        "scores": scores
    })
    print(f"{name}: {scores.round(3)} -> mean={scores.mean():.4f}, std={scores.std():.4f}")

Random Forest: [0.793 0.868 0.802 0.865 0.853] -> mean=0.8365, std=0.0323
XGBoost (plain): [0.829 0.873 0.804 0.881 0.882] -> mean=0.8537, std=0.0315
XGBoost (weighted): [0.813 0.879 0.821 0.883 0.883] -> mean=0.8558, std=0.0316
LightGBM (stable): [0.818 0.87  0.818 0.881 0.877] -> mean=0.8528, std=0.0287


In [42]:
cv_comparison = pd.DataFrame(cv_results)[["model", "mean_pr_auc", "std_pr_auc"]]
cv_comparison = cv_comparison.sort_values("mean_pr_auc", ascending=False).reset_index(drop=True)

In [43]:
cv_comparison

,model,mean_pr_auc,std_pr_auc
0,XGBoost (weighted),0.855833,0.031582
1,XGBoost (plain),0.853695,0.031537
2,LightGBM (stable),0.852759,0.028652
3,Random Forest,0.836499,0.032282


In [45]:
param_grid = {
    "n_estimators": [200, 300, 400, 500],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "max_depth": [3, 4, 5, 6, 8],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "scale_pos_weight": [1, spw / 2, spw]
}

In [46]:
xgb_base = XGBClassifier(random_state=42, n_jobs=-1, eval_metric="aucpr")

In [47]:
random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_grid,
    n_iter=30,
    scoring="average_precision",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=2
)

random_search.fit(X_train, y_train)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


RandomizedSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
                   estimator=XGBClassifier(base_score=None, booster=None,
                                           callbacks=None,
                                           colsample_bylevel=None,
                                           colsample_bynode=None,
                                           colsample_bytree=None, device=None,
                                           early_stopping_rounds=None,
                                           enable_categorical=True,
                                           eval_metric='aucpr',
                                           feature_types=None,
                                           feature_weights=None, gamma=None,
                                           gr...
                                           num_parallel_tree=None, ...),
                   n_iter=30, n_jobs=-1,
                   param_distributions={'colsample_bytree': [0.6, 0.8, 1.0],
                                        'learning_rate': [0.01, 0.03, 0.05,
                                                          0.1],
                                        'max_depth': [3, 4, 5, 6, 8],
                                        'n_estimators': [200, 300, 400, 500],
                                        'scale_pos_weight': [1,
                                                             np.float64(299.512084592145),
                                                             np.float64(599.02416918429)],
                                        'subsample': [0.6, 0.8, 1.0]},
                   random_state=42, scoring='average_precision', verbose=2)

In [48]:
print("Best PR-AUC (cross-validated):", random_search.best_score_)
print("Best parameters:")
for key, value in random_search.best_params_.items():
    print(f"  {key}: {value}")

xgb_tuned = random_search.best_estimator_

Best PR-AUC (cross-validated): 0.8615133215574751
Best parameters:
  subsample: 0.6
  scale_pos_weight: 599.02416918429
  n_estimators: 500
  max_depth: 8
  learning_rate: 0.1
  colsample_bytree: 1.0


In [49]:
results.append(evaluate_model("XGBoost (tuned)", xgb_tuned, X_test, y_test))

--- XGBoost (tuned) ---
Precision: 0.9569 | Recall: 0.7817 | F1: 0.8605 | PR-AUC: 0.8347
Confusion Matrix:
 [[84971     5]
 [   31   111]]



In [50]:
comparison_df = pd.DataFrame(results).sort_values("pr_auc", ascending=False).reset_index(drop=True)

In [51]:
comparison_df

,model,precision,recall,f1,pr_auc
0,XGBoost (tuned),0.956897,0.781690,0.860465,0.834750
1,XGBoost (plain),0.961905,0.711268,0.817814,0.824890
2,LightGBM (stabilised),0.938053,0.746479,0.831373,0.816054
3,XGBoost (scale_pos_weight),0.931034,0.760563,0.837209,0.813679
4,Random Forest (baseline),0.956897,0.781690,0.860465,0.803127
5,Isolation Forest (unsupervised),0.215278,0.218310,0.216783,0.108285
6,LightGBM (scale_pos_weight),0.027903,0.830986,0.053992,0.023474
7,LightGBM (plain),0.081395,0.049296,0.061404,0.005598


In [53]:
joblib.dump(xgb_tuned, "xgb_tuned.joblib")

['xgb_tuned.joblib']